# ERA5 archive 合併工具

這個 notebook 會把多個 ERA5 歷史資料資料夾合併成單一 `ERA5_archive`，
並檢查是否有欄位不一致或無法合併的格式問題。

- 來源資料夾：`ERA5_2013_20180325`、`ERA5_data _20180325_202303`、`ERA5_data _20230212_20250111`
- 目的資料夾：`ERA5_archive`
- 會針對同一氣象站的檔案合併（以檔名解析站號/站名/緯度/經度）
- 同一個時間點重複的資料會被去除（保留最先出現的紀錄）


In [ ]:
import os
from pathlib import Path
import pandas as pd

SOURCE_FOLDERS = [
    Path('..') / 'ERA5_2013_20180325',
    Path('..') / 'ERA5_data _20180325_202303',
    Path('..') / 'ERA5_data _20230212_20250111',
]
OUTPUT_FOLDER = Path('..') / 'ERA5_archive'

REQUIRED_COLUMNS = {
    'time',
    'temperature_2m',
    'relativehumidity_2m',
    'precipitation',
    'windspeed_10m',
    'winddirection_10m',
}

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


In [ ]:
def parse_station_info(path: Path):
    stem = path.stem
    parts = stem.split('_')
    if len(parts) < 4:
        raise ValueError(f'無法解析檔名：{path.name}')
    station_id = parts[0]
    lat = parts[-2]
    lon = parts[-1]
    name = '_'.join(parts[1:-2])
    return station_id, name, lat, lon


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'Wu' in df.columns and 'u' not in df.columns:
        df = df.rename(columns={'Wu': 'u'})
    if 'Wv' in df.columns and 'v' not in df.columns:
        df = df.rename(columns={'Wv': 'v'})
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
    return df


def load_station_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = normalize_columns(df)
    missing = REQUIRED_COLUMNS.difference(df.columns)
    if missing:
        raise ValueError(f'{path} 缺少欄位: {sorted(missing)}')
    return df


In [ ]:
all_files = []
for folder in SOURCE_FOLDERS:
    if not folder.exists():
        raise FileNotFoundError(f'找不到資料夾: {folder}')
    all_files.extend(sorted(folder.glob('*.csv')))

print(f'找到 {len(all_files)} 個 CSV 檔案')

column_mismatches = []
for path in all_files:
    try:
        df = load_station_file(path)
    except Exception as exc:
        column_mismatches.append((path, str(exc)))

if column_mismatches:
    print('以下檔案格式有問題，請先修正後再合併：')
    for path, error in column_mismatches:
        print(f'- {path}: {error}')
    raise SystemExit('格式檢查未通過')
else:
    print('格式檢查通過，開始合併')


In [ ]:
station_groups = {}
for path in all_files:
    station_key = parse_station_info(path)
    station_groups.setdefault(station_key, []).append(path)

print(f'共 {len(station_groups)} 個氣象站需要合併')


In [ ]:
merged_count = 0
for station_key, paths in station_groups.items():
    station_id, name, lat, lon = station_key
    dfs = [load_station_file(path) for path in paths]
    merged = pd.concat(dfs, ignore_index=True)
    merged['time'] = pd.to_datetime(merged['time'])
    merged = merged.drop_duplicates(subset=['time']).sort_values('time')
    output_name = f'{station_id}_{name}_{lat}_{lon}.csv'
    output_path = OUTPUT_FOLDER / output_name
    merged.to_csv(output_path, index=False)
    merged_count += 1

print(f'合併完成，共輸出 {merged_count} 個站點檔案到 {OUTPUT_FOLDER}')
